# Setup


In [14]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("../data")

## Load


In [15]:
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")

## Convert dates


In [ ]:
#This will convert each column from the text into the real date

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])



## Inspect


In [17]:
print("orders shape:", orders.shape)
print("duplicated rows:", orders.duplicated().sum())
print("duplicated order ids:", orders["order_id"].duplicated().sum())
print("null values by column:")
print(orders.isna().sum())
print("order status counts:")
print(orders["order_status"].value_counts(dropna=False))

orders shape: (99441, 8)
duplicated rows: 0
duplicated order ids: 0
null values by column:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## Anomaly checks


In [18]:
delivered_before_purchase = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_delivered_customer_date"].notna()
    & (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"])
].copy()

estimate_before_purchase = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_estimated_delivery_date"].notna()
    & (orders["order_estimated_delivery_date"] < orders["order_purchase_timestamp"])
].copy()

delivered_no_date = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_delivered_customer_date"].isna()
].copy()

print("delivered before purchase:", len(delivered_before_purchase))
print("estimated delivery before purchase:", len(estimate_before_purchase))
print("delivered with no delivery date:", len(delivered_no_date))

delivered before purchase: 0
estimated delivery before purchase: 0
delivered with no delivery date: 8


## Cleaning


In [19]:
delivered = orders.loc[orders["order_status"].eq("delivered")].copy()

anomaly_indices = (
    delivered_before_purchase.index
    .union(estimate_before_purchase.index)
    .union(delivered_no_date.index)
)
orders_clean = delivered.loc[~delivered.index.isin(anomaly_indices)].copy()

cleaning_log = [
    ("orders raw (date-converted)", len(orders)),
    ("delivered after status filter", len(delivered)),
    ("delivered before purchase", len(delivered_before_purchase)),
    ("estimated delivery before purchase", len(estimate_before_purchase)),
    ("delivered with no delivery date", len(delivered_no_date)),
    ("orders_clean after anomaly removals", len(orders_clean)),
]

print("Cleaning row-count log:")
for stage, row_count in cleaning_log:
    print(f"{stage}: {row_count}")

Cleaning row-count log:
orders raw (date-converted): 99441
delivered after status filter: 96478
delivered before purchase: 0
estimated delivery before purchase: 0
delivered with no delivery date: 8
orders_clean after anomaly removals: 96470
